# VisualCOT-Swap — Full Pilot
**LLaVA-1.5-7B → InternVL2-8B → Idefics2-8B · 20 samples each**

Run all cells top to bottom. Auto-detects Colab / Lightning AI / local GPU. No manual config needed.

In [ ]:
# ── CELL 1: Environment detection + install ───────────────────────────────────
import subprocess, sys, os, glob
from pathlib import Path

# ── Detect platform ───────────────────────────────────────────────────────────
ON_COLAB     = 'google.colab' in sys.modules or Path('/content').exists()
ON_LIGHTNING = Path('/home/zeus').exists() or 'zeus' in os.environ.get('HOME', '')
ON_KAGGLE    = Path('/kaggle').exists()
PLATFORM     = 'colab' if ON_COLAB else 'lightning' if ON_LIGHTNING else 'kaggle' if ON_KAGGLE else 'local'
print(f'Platform : {PLATFORM}')

# ── Fix CUDA library path before installing bitsandbytes ─────────────────────
for pat in ['/usr/local/cuda*/lib64', '/usr/lib/x86_64-linux-gnu', '/opt/cuda/lib64']:
    hits = glob.glob(pat)
    if hits:
        lib = hits[0]
        cur = os.environ.get('LD_LIBRARY_PATH', '')
        if lib not in cur:
            os.environ['LD_LIBRARY_PATH'] = f'{lib}:{cur}'
        print(f'LD_LIBRARY_PATH += {lib}')
        break

# ── Install ───────────────────────────────────────────────────────────────────
pkgs = [
    'transformers==4.47.0', 'accelerate==0.27.0', 'bitsandbytes',
    'datasets==2.20.0', 'Pillow', 'scipy', 'scikit-learn',
    'matplotlib', 'seaborn', 'opencv-python-headless', 'spacy', 'tqdm',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U'] + pkgs, check=True)
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm', '-q'], check=True)

# ── Verify bitsandbytes ───────────────────────────────────────────────────────
r = subprocess.run([sys.executable, '-c', 'import bitsandbytes; print(bitsandbytes.__version__)'],
                   capture_output=True, text=True)
BNB_OK = r.returncode == 0
print(f'bitsandbytes : {r.stdout.strip() if BNB_OK else "BROKEN — will use fp16 fallback"}')
print('Install done.')

In [ ]:
# ── CELL 2: Clone repo + locate root ─────────────────────────────────────────
import os, sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/jeeth-kataria/CoherenceTax.git'

# Pick a writable clone dir based on platform
for _d in ['/content/visualcot', '/home/zeus/visualcot', '/tmp/visualcot', str(Path.home() / 'visualcot')]:
    if Path(_d).parent.exists():
        CLONE_DIR = _d
        break

if not Path(CLONE_DIR).exists():
    subprocess.run(['git', 'clone', REPO_URL, CLONE_DIR], check=True)
else:
    subprocess.run(['git', '-C', CLONE_DIR, 'pull'], check=True)

def find_repo_root() -> Path:
    env = os.environ.get('COHERENCE_TAX_ROOT') or os.environ.get('REPO_DIR')
    if env:
        p = Path(env).expanduser().resolve()
        if (p / 'src').exists():
            return p
    search = [Path(CLONE_DIR), Path.cwd(),
              Path('/content'), Path('/home/zeus'), Path('/tmp'), Path('/workspace'),
              Path.home()]
    seen = set()
    for root in search:
        if not root.exists():
            continue
        for c in [root, *root.parents]:
            if c in seen:
                continue
            seen.add(c)
            if (c / 'src').exists() and (c / 'scripts').exists() and (c / 'requirements.txt').exists():
                return c.resolve()
            try:
                for marker in c.glob('**/src/models/loader.py'):
                    return marker.parents[2].resolve()
            except Exception:
                pass
    raise FileNotFoundError('Repo not found. Set COHERENCE_TAX_ROOT env var to repo path.')

REPO_DIR    = str(find_repo_root())
RESULTS_DIR = f'{REPO_DIR}/results'
LOCAL_DATA  = f'{REPO_DIR}/data'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Repo    : {REPO_DIR}')
print(f'Results : {RESULTS_DIR}')

In [ ]:
# ── CELL 3: Download ScienceQA ────────────────────────────────────────────────
import json
from pathlib import Path

sqa_meta = Path(f'{LOCAL_DATA}/scienceqa/metadata.json')

if sqa_meta.exists():
    print(f'ScienceQA cached: {len(json.loads(sqa_meta.read_text()))} records')
else:
    r = subprocess.run(
        [sys.executable, 'scripts/prepare_data.py',
         '--output', LOCAL_DATA, '--max_scienceqa', '8000', '--max_gqa', '0'],
        capture_output=True, text=True
    )
    print(r.stdout[-2000:])
    if r.returncode != 0:
        print('STDERR:', r.stderr[-800:])

In [ ]:
# ── CELL 4: Build benchmark ───────────────────────────────────────────────────
LOCAL_BM = Path(f'{LOCAL_DATA}/swap_pairs/benchmark_metadata.json')

if LOCAL_BM.exists():
    print(f'Benchmark cached: {len(json.loads(LOCAL_BM.read_text()))} triplets')
else:
    r = subprocess.run(
        [sys.executable, 'scripts/build_benchmark.py',
         '--n', '20', '--output', str(LOCAL_BM.parent), '--dry_run'],
        capture_output=True, text=True
    )
    print(r.stdout[-2000:])
    if r.returncode != 0:
        print('STDERR:', r.stderr[-800:])

LOCAL_BM = str(LOCAL_BM)

In [ ]:
# ── CELL 5: Imports + shared setup ───────────────────────────────────────────
import gc, json, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
from datasets import load_dataset
from tqdm.notebook import tqdm

from src.models.loader       import load_model_and_processor, build_prompt, parse_steps, unload_model, load_config, detect_env
from src.attribution.rollout import attention_rollout, find_image_token_range
from src.metrics.gfs         import compute_gfs_sequence, summarize_result
from src.utils.stats         import save_checkpoint, load_checkpoint

CFG         = load_config()
ENV         = detect_env()
DEVICE      = 'cuda' if ENV['has_gpu'] else 'cpu'
IMG_SIZE    = CFG['attribution']['image_size']
MAX_STEPS   = CFG['gfs']['max_steps']
MAX_NEW_TOK = 300
N_SAMPLES   = 20

print(f'Platform : {ENV["platform"]}')
print(f'GPU      : {ENV["gpu_name"]}')
print(f'VRAM     : {ENV["vram_gb"]} GB')
print(f'CUDA     : {ENV["cuda_version"]}')
print(f'Device   : {DEVICE}')

print('\nLoading ScienceQA...')
SQA_DS  = load_dataset('derek-thomas/ScienceQA', split='train', trust_remote_code=True)
TRIPLETS = json.loads(open(LOCAL_BM).read())
print(f'ScienceQA : {len(SQA_DS)} rows')
print(f'Benchmark : {len(TRIPLETS)} triplets')

def get_image(hf_index):
    img = SQA_DS[hf_index].get('image')
    return img.convert('RGB') if isinstance(img, Image.Image) else None

def run_one(model, processor, model_cfg, model_key, triplet):
    orig      = triplet['original']
    swap_meta = triplet.get('swap')
    question  = orig['question']
    orig_img  = get_image(orig['hf_index'])
    if orig_img is None:
        return None
    swap_img = get_image(swap_meta['hf_index']) if swap_meta else None

    n_patches = model_cfg.get('n_image_patches', 576)
    tok_id    = model_cfg.get('image_token_id', -200)

    def get_gfs(img):
        prompt = build_prompt(model_key, processor, question, cot=True)
        inputs = processor(text=prompt, images=img, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOK, do_sample=False)
        text  = processor.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        steps = parse_steps(text)[:MAX_STEPS]
        s, e  = find_image_token_range(inputs['input_ids'], tok_id, n_patches)
        hms   = [
            attention_rollout(
                model,
                processor(text=build_prompt(model_key, processor, st, cot=False),
                          images=img, return_tensors='pt').to(DEVICE),
                s, e, IMG_SIZE
            )
            for st in steps
        ]
        return text, steps, compute_gfs_sequence(hms, steps)

    cot_text, steps, gfs_orig = get_gfs(orig_img)
    _, _, gfs_swap = get_gfs(swap_img) if swap_img else (None, [], None)

    res = {
        'triplet_id': triplet['triplet_id'], 'model_key': model_key,
        'question': question, 'subject': orig.get('subject', ''),
        'cot_text': cot_text, 'steps': steps, 'n_steps': len(steps),
        'gfs_per_step': gfs_orig, 'gfs_per_step_swapped': gfs_swap,
    }
    res.update(summarize_result(res))
    return res

print('Ready.')

In [ ]:
# ── CELL 6: run_pilot() ───────────────────────────────────────────────────────
def run_pilot(model_key):
    out_dir = Path(f'{RESULTS_DIR}/pilot/{model_key}')
    out_dir.mkdir(parents=True, exist_ok=True)
    ckpt    = str(out_dir / 'checkpoint.json')

    completed = load_checkpoint(ckpt)
    done_ids  = {r['triplet_id'] for r in completed}
    todo      = [t for t in TRIPLETS if t['triplet_id'] not in done_ids][:N_SAMPLES]

    if not todo:
        print(f'{model_key}: already done ({len(completed)} results)')
        return completed
    if completed:
        print(f'{model_key}: resuming — {len(completed)} done, {len(todo)} left')

    print(f'\n{"="*52}\n  {model_key}  |  {len(todo)} samples'
          f'  |  VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB\n{"="*52}')

    model, processor, model_cfg = load_model_and_processor(model_key)
    results = list(completed)
    errors  = 0
    t0      = time.time()

    for i, triplet in enumerate(tqdm(todo, desc=model_key)):
        try:
            res = run_one(model, processor, model_cfg, model_key, triplet)
            if res:
                results.append(res)
                (out_dir / f"{res['triplet_id']}.json").write_text(json.dumps(res, default=str))
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            errors += 1
            print(f'OOM — skipped {triplet["triplet_id"]}')
        except Exception as e:
            errors += 1
            print(f'Error [{triplet["triplet_id"]}]: {e}')

        if (i + 1) % 5 == 0:
            save_checkpoint(results, ckpt)
            eta = ((time.time() - t0) / (i + 1)) * (len(todo) - i - 1)
            print(f'  checkpoint [{i+1}/{len(todo)}]  ETA {eta/60:.1f} min')

    save_checkpoint(results, ckpt)

    valid_gfs    = [r['gfs_mean']        for r in results if r.get('gfs_mean')        is not None]
    valid_steps  = [r['n_steps']         for r in results]
    valid_slopes = [r['gfs_decay_slope'] for r in results if r.get('gfs_decay_slope') is not None]

    summary = {
        'model':             model_key,
        'n_samples':         len(results),
        'n_errors':          errors,
        'avg_gfs':           round(float(np.mean(valid_gfs)),    4) if valid_gfs    else None,
        'avg_steps':         round(float(np.mean(valid_steps)),  2) if valid_steps  else None,
        'avg_decay_slope':   round(float(np.mean(valid_slopes)), 4) if valid_slopes else None,
        'samples_ge3_steps': sum(1 for s in valid_steps if s >= 3),
        'elapsed_min':       round((time.time() - t0) / 60, 1),
        'env':               ENV,
    }
    (out_dir / 'pilot_summary.json').write_text(json.dumps(summary, indent=2))

    unload_model(model)
    gc.collect()

    print(f'\n✓ {model_key} done in {summary["elapsed_min"]} min')
    print(f'  avg_steps={summary["avg_steps"]}  avg_gfs={summary["avg_gfs"]}  slope={summary["avg_decay_slope"]}')
    return results

print('run_pilot() ready')

In [ ]:
# ── CELL 7: LLaVA-1.5-7B ─────────────────────────────────────────────────────
results_llava = run_pilot('llava')

In [ ]:
# ── CELL 8: InternVL2-8B ──────────────────────────────────────────────────────
results_internvl = run_pilot('internvl')

In [ ]:
# ── CELL 9: Idefics2-8B ───────────────────────────────────────────────────────
results_idefics2 = run_pilot('idefics2')

In [ ]:
# ── CELL 10: Results + plot ───────────────────────────────────────────────────
MODEL_COLORS = {'llava': '#534AB7', 'internvl': '#1D9E75', 'idefics2': '#D85A30'}
MODEL_LABELS = {'llava': 'LLaVA-1.5-7B', 'internvl': 'InternVL2-8B', 'idefics2': 'Idefics2-8B'}
all_results  = {'llava': results_llava, 'internvl': results_internvl, 'idefics2': results_idefics2}

# Summary table
print(f'{"="*62}')
print(f'{"Model":<20} {"avg_steps":>9} {"avg_gfs":>8} {"slope":>8} {">= 3steps":>9}')
print(f'{"-"*62}')
for mk in ['llava', 'internvl', 'idefics2']:
    sp = Path(f'{RESULTS_DIR}/pilot/{mk}/pilot_summary.json')
    if not sp.exists():
        print(f'{mk:<20}  NO DATA'); continue
    s = json.loads(sp.read_text())
    print(f'{MODEL_LABELS[mk]:<20} {str(s["avg_steps"]):>9} {str(s["avg_gfs"]):>8} '
          f'{str(s["avg_decay_slope"]):>8} {s["samples_ge3_steps"]}/{s["n_samples"]:>7}')
print(f'{"="*62}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
for mk, results in all_results.items():
    if not results: continue
    by_step = [[] for _ in range(MAX_STEPS)]
    for r in results:
        for k, g in enumerate((r.get('gfs_per_step') or [])[:MAX_STEPS]):
            if g is not None: by_step[k].append(g)
    means = [np.mean(s) if s else np.nan for s in by_step]
    sems  = [np.std(s)/np.sqrt(len(s)) if len(s)>1 else 0 for s in by_step]
    valid = [i for i,m in enumerate(means) if not np.isnan(m)]
    if not valid: continue
    x = np.array(valid)+1; y = np.array([means[i] for i in valid]); e = np.array([sems[i] for i in valid])
    ax.plot(x, y, 'o-', color=MODEL_COLORS[mk], label=MODEL_LABELS[mk], markersize=5)
    ax.fill_between(x, y-e, y+e, color=MODEL_COLORS[mk], alpha=0.15)
ax.set_xlabel('Step k'); ax.set_ylabel('Mean GFS'); ax.set_ylim(0,1)
ax.legend(frameon=False); ax.set_title('GFS Decay — Pilot (n=20)')
ax.spines[['top','right']].set_visible(False)

ax2 = axes[1]
mk_list = [mk for mk in ['llava','internvl','idefics2'] if all_results.get(mk)]
by = [np.mean([r['gfs_mean'] for r in all_results[mk] if r.get('gfs_mean') is not None] or [0]) for mk in mk_list]
be = [np.std( [r['gfs_mean'] for r in all_results[mk] if r.get('gfs_mean') is not None] or [0]) for mk in mk_list]
ax2.bar(range(len(mk_list)), by, yerr=be, color=[MODEL_COLORS[mk] for mk in mk_list], capsize=4, alpha=0.8)
ax2.set_xticks(range(len(mk_list)))
ax2.set_xticklabels([MODEL_LABELS[mk] for mk in mk_list], rotation=10, ha='right')
ax2.set_ylabel('Mean GFS'); ax2.set_ylim(0,1); ax2.set_title('Mean GFS by Model')
ax2.spines[['top','right']].set_visible(False)

plt.tight_layout()
out_png = f'{RESULTS_DIR}/pilot/pilot_combined.png'
fig.savefig(out_png, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out_png}')

In [ ]:
# ── CELL 11: Qualitative check + diagnosis ────────────────────────────────────
if results_llava:
    r = results_llava[0]
    print(f'Q        : {r["question"]}')
    print(f'Steps    : {r["n_steps"]}')
    print(f'GFS/step : {[round(g,3) if g is not None else None for g in r["gfs_per_step"]]}')
    print(f'Slope    : {r.get("gfs_decay_slope")}')
    print(f'Swap sens: {r.get("swap_sensitivity_score")}')
    print(f'\nCoT:\n{r["cot_text"][:500]}')

print('\n── DIAGNOSIS ──')
for mk in ['llava','internvl','idefics2']:
    results = all_results.get(mk, [])
    if not results: print(f'  {mk}: NO DATA'); continue
    all_gfs = [g for r in results for g in (r.get('gfs_per_step') or []) if g is not None]
    if not all_gfs:
        print(f'  {mk}: all GFS=None  →  output_attentions not firing; switch attribution to gradcam')
    elif max(all_gfs) - min(all_gfs) < 0.05:
        print(f'  {mk}: GFS flat ({min(all_gfs):.3f}–{max(all_gfs):.3f})  →  heatmaps uniform; check rollout impl')
    else:
        print(f'  {mk}: GFS varies ({min(all_gfs):.3f}–{max(all_gfs):.3f})  ✓')